# 02 — Medallion Benchmark: Dataproc Lightning Engine (Base & Gluten/Velox C++ NQE)

This notebook benchmarks **Dataproc Lightning Engine** on Google Cloud Compute Engine (`2.3-debian12`, x86 `n2d-standard` + NVMe Local SSDs) across two sequential passes over the **exact same open GCS Parquet Bronze tables** (`gs://${BENCH_BUCKET}/bronze/${SCALE_PRESET}/`) used in Notebook `01`:

1. **Pass 1 — `spark_lightning_base` (`spark.dataproc.lightningEngine.runtime=default`):**
   Lightning Engine cluster with Native Query Execution (NQE) disabled (`nqe_enabled = False`).
2. **Pass 2 — `spark_lightning_nqe` (`spark.dataproc.lightningEngine.runtime=native`):**
   Lightning Engine with **Apache Gluten + Meta/Apache Velox C++ Native Query Execution** (`nqe_enabled = True`).

### Non-Negotiable Fairness Guardrails (`SPEC.md` §1.2 & §1.3)
- **Pure GCS Parquet I/O:** Zero BigQuery Spark connector reads or query pushdown. All Bronze reads and Silver/Gold writes execute purely against `gs://${BENCH_BUCKET}/...`.
- **x86 Hardware Parity:** `CLUSTER_LGT_NAME` uses identical VM family, vCPU count, RAM, and NVMe Local SSD configuration as `CLUSTER_STD_NAME` (`01`), differing only in `--engine=lightning`.
- **Shared Medallion Runner:** Submits `<repo_root>/scripts/run_spark_medallion.py` (with an identical `SPEC.md` §3 self-contained fallback only when run standalone prior to repository integration).

### Key Deliverables in This Notebook
- Sequential execution and telemetry logging (`results/live_runs/benchmark_runs.jsonl` & BigQuery `${DS_METRICS}.benchmark_runs`) for both `spark_lightning_base` and `spark_lightning_nqe`.
- **Catalyst vs. Gluten/Velox Physical Plan Audit:** Parses `df.explain("formatted")` across Stages `1`, `2A`, `2B`, and `3`, contrasting native C++ operators (`*ExecTransformer`, `NativeFileScan`, `HashAggregateTransformer`, `WindowExecTransformer`) with the Stage `2B` `@F.udf` fallback boundary (`VeloxColumnarToRow -> BatchEvalPythonExec -> RowToVeloxColumnar`).
- **Parity, Speedup & Surcharge Break-Even:** Verifies 100% row-count parity and `±0.01%` `checksum_net_paid` tolerance, quantifies the Stage `2B` UDF penalty vs. Stage `2A`, and evaluates measured speedup against the `+$0.0025/vCPU-hr` Lightning Engine surcharge.

## 1. Environment Configuration & Cross-Worker Setup

In [ ]:
import inspect
import json
import os
import re
import subprocess
import sys
import tempfile
from contextlib import nullcontext
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj: Any) -> None:
        print(obj)


def find_repo_root(start: Path | None = None) -> Path:
    '''Walk upward from cwd to locate the repository root containing SPEC.md.'''
    origin = (start or Path.cwd()).resolve()
    for candidate in (origin, *origin.parents):
        if (candidate / "SPEC.md").is_file():
            return candidate
    raise FileNotFoundError("Could not locate repository root containing SPEC.md.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def _load_env_file_fallback(env_path: Path) -> None:
    '''Minimal parser for .env files using `export KEY="VALUE"` syntax.'''
    if not env_path.is_file():
        return
    pattern = re.compile(r'^(?:export\s+)?([A-Za-z_][A-Za-z0-9_]*)=(.*)$')
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        match = pattern.match(line)
        if not match:
            continue
        key, val = match.group(1), match.group(2).strip()
        if (val.startswith('"') and val.endswith('"')) or (val.startswith("'") and val.endswith("'")):
            val = val[1:-1]
        val = re.sub(r"\$\{([A-Za-z_][A-Za-z0-9_]*)\}", lambda m: os.environ.get(m.group(1), ""), val)
        os.environ.setdefault(key, val)


def _load_dotenv_safe(env_path: Path) -> None:
    if not env_path.is_file():
        return
    try:
        from dotenv import load_dotenv

        load_dotenv(env_path, override=False)
    except ImportError:
        pass
    _load_env_file_fallback(env_path)


def run_gcloud_text(args: list[str]) -> str:
    '''Run gcloud and strip Cloudtop mTLS certificate-proxy preamble lines.'''
    proc = subprocess.run(["gcloud", *args], check=True, text=True, capture_output=True)
    lines = [ln for ln in proc.stdout.splitlines() if not ln.startswith("Created ")]
    return "\n".join(lines).strip()


def run_gcloud_json(args: list[str]) -> Any:
    '''Run gcloud --format=json and parse starting from the first JSON token.'''
    proc = subprocess.run(["gcloud", *args, "--format=json"], check=True, text=True, capture_output=True)
    raw = proc.stdout
    indices = [idx for idx in (raw.find("{"), raw.find("[")) if idx >= 0]
    if not indices:
        raise ValueError(f"Expected JSON output from gcloud {' '.join(args)}, got: {raw[:200]}")
    return json.loads(raw[min(indices):])


def parse_machine_vcpus(machine_type: str) -> int:
    '''Extract vCPU count from standard GCP GCE machine type names.'''
    custom_match = re.search(r"-custom-(\d+)-\d+$", machine_type)
    if custom_match:
        return int(custom_match.group(1))
    std_match = re.search(r"-(\d+)$", machine_type)
    if std_match:
        return int(std_match.group(1))
    raise ValueError(f"Unable to parse vCPU count from WORKER_MACHINE_TYPE='{machine_type}'.")


_load_dotenv_safe(REPO_ROOT / ".env")

try:
    from scripts.metrics_helper import load_benchmark_runs, log_stage_metric, resolve_bench_config

    METRICS_HELPER_AVAILABLE = True
    try:
        BENCH_CFG = resolve_bench_config()
    except Exception:
        BENCH_CFG = {}
except ImportError:
    METRICS_HELPER_AVAILABLE = False
    BENCH_CFG = {}


def _cfg(key: str, default: str = "") -> str:
    val = os.environ.get(key)
    if val is not None and val != "":
        return val
    cfg_val = BENCH_CFG.get(key) if isinstance(BENCH_CFG, dict) else None
    if cfg_val is not None and str(cfg_val) != "":
        return str(cfg_val)
    return default


PROJECT_ID = _cfg("PROJECT_ID")
if not PROJECT_ID or PROJECT_ID == "your-gcp-project-id":
    raise ValueError("PROJECT_ID must be set in .env or environment (cannot be placeholder 'your-gcp-project-id').")

PROJECT_NUMBER = _cfg("PROJECT_NUMBER")
if not PROJECT_NUMBER:
    PROJECT_NUMBER = run_gcloud_text(["projects", "describe", PROJECT_ID, "--format=value(projectNumber)"])

BENCH_BUCKET = _cfg("BENCH_BUCKET") or f"lakehouse-bench-{PROJECT_NUMBER}"
BENCH_BUCKET = BENCH_BUCKET.removeprefix("gs://").rstrip("/")

SCALE_PRESET = _cfg("SCALE_PRESET", "small_1g")
if SCALE_PRESET not in {"small_1g", "medium_100g", "large_500g"}:
    raise ValueError(f"Unsupported SCALE_PRESET='{SCALE_PRESET}'. Expected small_1g, medium_100g, or large_500g.")

DATAPROC_REGION = _cfg("DATAPROC_REGION", "us-central1")
DATAPROC_ZONE = _cfg("DATAPROC_ZONE", "us-central1-a")
DATAPROC_IMAGE_VERSION = _cfg("DATAPROC_IMAGE_VERSION", "2.3-debian12")
CLUSTER_LGT_NAME = _cfg("CLUSTER_LGT_NAME", "bench-spark-lightning-x86")
MASTER_MACHINE_TYPE = _cfg("MASTER_MACHINE_TYPE", "n2d-standard-4")
WORKER_MACHINE_TYPE = _cfg("WORKER_MACHINE_TYPE", "n2d-standard-8")
NUM_PRIMARY_WORKERS = int(_cfg("NUM_PRIMARY_WORKERS", "2"))
NUM_WORKER_LOCAL_SSDS = int(_cfg("NUM_WORKER_LOCAL_SSDS", "1"))
WORKER_VCPUS_PER_NODE = parse_machine_vcpus(WORKER_MACHINE_TYPE)
TOTAL_WORKER_VCPUS = WORKER_VCPUS_PER_NODE * NUM_PRIMARY_WORKERS

RUN_PYTHON_UDF_STAGE = _cfg("RUN_PYTHON_UDF_STAGE", "true").lower() in {"1", "true", "yes"}
UDF_MAX_SAMPLE_ROWS = int(_cfg("UDF_MAX_SAMPLE_ROWS", "500000"))
DS_METRICS = _cfg("DS_METRICS", "bench_metrics")
WRITE_METRICS_TO_BQ = _cfg("WRITE_METRICS_TO_BQ", "true").lower() in {"1", "true", "yes"}
EXECUTION_MODE = _cfg("SPARK_EXECUTION_MODE", "dataproc_submit")
RUN_ID = _cfg("RUN_ID") or f"{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}_{SCALE_PRESET}"

LIVE_JSONL_PATH = REPO_ROOT / "results" / "live_runs" / "benchmark_runs.jsonl"

config_summary = pd.DataFrame(
    [
        ("RUN_ID", RUN_ID),
        ("SCALE_PRESET", SCALE_PRESET),
        ("BENCH_BUCKET", f"gs://{BENCH_BUCKET}"),
        ("CLUSTER_LGT_NAME", CLUSTER_LGT_NAME),
        ("DATAPROC_REGION / ZONE", f"{DATAPROC_REGION} / {DATAPROC_ZONE}"),
        ("DATAPROC_IMAGE_VERSION", DATAPROC_IMAGE_VERSION),
        ("WORKER_MACHINE_TYPE", f"{WORKER_MACHINE_TYPE} ({WORKER_VCPUS_PER_NODE} vCPUs/node)"),
        ("NUM_PRIMARY_WORKERS", NUM_PRIMARY_WORKERS),
        ("TOTAL_WORKER_VCPUS", TOTAL_WORKER_VCPUS),
        ("NUM_WORKER_LOCAL_SSDS", NUM_WORKER_LOCAL_SSDS),
        ("RUN_PYTHON_UDF_STAGE", RUN_PYTHON_UDF_STAGE),
        ("UDF_MAX_SAMPLE_ROWS", UDF_MAX_SAMPLE_ROWS),
        ("WRITE_METRICS_TO_BQ", WRITE_METRICS_TO_BQ),
        ("METRICS_HELPER_AVAILABLE", METRICS_HELPER_AVAILABLE),
        ("EXECUTION_MODE", EXECUTION_MODE),
    ],
    columns=["Setting", "Value"],
)
display(config_summary)

## 2. Pre-Flight Hardware, Engine & Bronze GCS Parquet Readiness

Verifies two invariants before starting any timed run:
1. `${CLUSTER_LGT_NAME}` is `RUNNING` with `config.engine == "LIGHTNING"`, image `2.3-debian12`, x86 workers matching `WORKER_MACHINE_TYPE × NUM_PRIMARY_WORKERS`, and `NUM_WORKER_LOCAL_SSDS` NVMe disks.
2. All 8 Bronze TPC-DS Parquet folders exist under `gs://${BENCH_BUCKET}/bronze/${SCALE_PRESET}/<table>/`.

In [ ]:
REQUIRED_BRONZE_TABLES = (
    "store_sales",
    "store_returns",
    "web_sales",
    "web_returns",
    "customer",
    "customer_demographics",
    "item",
    "date_dim",
)


def verify_lightning_cluster() -> dict[str, Any]:
    '''Validate CLUSTER_LGT_NAME exists, is RUNNING, and uses Lightning + x86 parity.'''
    cluster = run_gcloud_json(
        [
            "dataproc",
            "clusters",
            "describe",
            CLUSTER_LGT_NAME,
            f"--region={DATAPROC_REGION}",
            f"--project={PROJECT_ID}",
        ]
    )
    state = cluster.get("status", {}).get("state", "UNKNOWN")
    cfg = cluster.get("config", {})
    engine = cfg.get("engine", "DEFAULT")
    image_version = cfg.get("softwareConfig", {}).get("imageVersion", "")
    worker_cfg = cfg.get("workerConfig", {})
    machine_uri = worker_cfg.get("machineTypeUri", "")
    machine_short = machine_uri.rsplit("/", 1)[-1] if machine_uri else ""
    num_workers = int(worker_cfg.get("numInstances", 0))
    num_ssds = int(worker_cfg.get("diskConfig", {}).get("numLocalSsds", 0))

    assert state == "RUNNING", f"Cluster {CLUSTER_LGT_NAME} state is '{state}', expected 'RUNNING'."
    assert engine == "LIGHTNING", f"Cluster {CLUSTER_LGT_NAME} engine is '{engine}', expected 'LIGHTNING'."
    assert not machine_short.startswith(("t2a-", "c4a-")), (
        f"ARM machine type '{machine_short}' violates x86 hardware parity (SPEC.md §1.3)."
    )
    assert machine_short == WORKER_MACHINE_TYPE, (
        f"Worker machine mismatch: cluster has '{machine_short}', expected '{WORKER_MACHINE_TYPE}'."
    )
    assert num_workers == NUM_PRIMARY_WORKERS, (
        f"Worker count mismatch: cluster has {num_workers}, expected {NUM_PRIMARY_WORKERS}."
    )
    assert num_ssds == NUM_WORKER_LOCAL_SSDS, (
        f"Local SSD mismatch: cluster has {num_ssds}, expected {NUM_WORKER_LOCAL_SSDS}."
    )
    return {
        "cluster_name": CLUSTER_LGT_NAME,
        "state": state,
        "engine": engine,
        "image_version": image_version,
        "worker_machine_type": machine_short,
        "num_workers": num_workers,
        "num_local_ssds": num_ssds,
    }


def verify_bronze_parquet_tables() -> pd.DataFrame:
    '''Confirm all 8 TPC-DS Bronze Parquet prefixes exist in GCS for SCALE_PRESET.'''
    from google.cloud import storage

    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(BENCH_BUCKET)
    rows: list[dict[str, Any]] = []
    missing: list[str] = []
    for table in REQUIRED_BRONZE_TABLES:
        prefix = f"bronze/{SCALE_PRESET}/{table}/"
        blobs = [
            b
            for b in bucket.list_blobs(prefix=prefix)
            if not b.name.endswith("/") and (b.name.endswith(".parquet") or (b.size or 0) > 0)
        ]
        total_bytes = sum(int(b.size or 0) for b in blobs)
        if not blobs:
            missing.append(f"gs://{BENCH_BUCKET}/{prefix}")
        rows.append(
            {
                "table": table,
                "gcs_prefix": f"gs://{BENCH_BUCKET}/{prefix}",
                "parquet_objects": len(blobs),
                "total_mb": round(total_bytes / (1024 * 1024), 2),
            }
        )
    if missing:
        raise FileNotFoundError(
            "Missing Bronze Parquet tables (run notebooks/00_data_copy_ingestion.ipynb first): "
            + ", ".join(missing)
        )
    return pd.DataFrame(rows)


if EXECUTION_MODE == "dataproc_submit":
    cluster_audit = verify_lightning_cluster()
    display(pd.DataFrame([cluster_audit]))
else:
    print(f"EXECUTION_MODE={EXECUTION_MODE}: skipping remote cluster describe.")

bronze_audit_df = verify_bronze_parquet_tables()
display(bronze_audit_df)

## 3. Shared PySpark Runner Resolution & Submission Helpers

- When `<repo_root>/scripts/run_spark_medallion.py` is present (the standard state on `main`), this notebook submits that exact shared script to `CLUSTER_LGT_NAME` so `01` and `02` execute identical PySpark logic.
- If `scripts/run_spark_medallion.py` is absent (for example, standalone branch testing), `resolve_runner_script()` materializes a clean temporary copy implementing the exact `SPEC.md` §3 Stages 1, 2A, 2B, and 3 logic without altering repository files.
- **Flag syntax note:** `gcloud dataproc jobs submit pyspark --properties` takes `KEY=VALUE` **without** a `spark:` prefix:
  - Pass 1: `--properties=spark.dataproc.lightningEngine.runtime=default`
  - Pass 2: `--properties=spark.dataproc.lightningEngine.runtime=native`

In [ ]:
FALLBACK_SPARK_RUNNER_CODE = r'''# Self-contained SPEC.md §3 PySpark Medallion runner (fallback when scripts/run_spark_medallion.py is absent).

from __future__ import annotations

import argparse
from contextlib import redirect_stdout
import io
import json
import time
from typing import Any

from pyspark.sql import DataFrame, SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType

VALID_ENGINES = ("spark_standard", "spark_lightning_base", "spark_lightning_nqe")
VALID_SCALES = ("small_1g", "medium_100g", "large_500g")


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Execute TPC-DS Medallion benchmark on Spark.")
    parser.add_argument("--engine-variant", required=True, choices=VALID_ENGINES)
    parser.add_argument("--scale-preset", required=True, choices=VALID_SCALES)
    parser.add_argument("--bench-bucket", required=True)
    parser.add_argument("--run-id", required=True)
    parser.add_argument("--worker-machine-type", default="n2d-standard-8")
    parser.add_argument("--total-worker-vcpus", type=int, default=16)
    parser.add_argument("--run-python-udf", default="true")
    parser.add_argument("--udf-max-sample-rows", type=int, default=500000)
    parser.add_argument("--metrics-gcs-uri", required=True)
    return parser.parse_args(argv)


def capture_formatted_plan(spark: SparkSession, df: DataFrame) -> str:
    try:
        explain_mode = spark._jvm.org.apache.spark.sql.execution.ExplainMode.fromString("formatted")
        return str(df._jdf.queryExecution().explainString(explain_mode))
    except Exception:
        buf = io.StringIO()
        with redirect_stdout(buf):
            df.explain(mode="formatted")
        return buf.getvalue()


def measure_stage_output(
    spark: SparkSession, path: str, checksum_col: str
) -> tuple[int, float]:
    df_out = spark.read.parquet(path)
    row = df_out.agg(
        F.count(F.lit(1)).alias("cnt"),
        F.round(F.sum(F.col(checksum_col).cast("double")), 2).alias("chk"),
    ).collect()[0]
    cnt = int(row["cnt"] or 0)
    chk = float(row["chk"] or 0.0)
    return cnt, chk


def build_stage1_silver(spark: SparkSession, bronze_root: str) -> DataFrame:
    ss = spark.read.parquet(f"{bronze_root}/store_sales/").filter(
        F.col("ss_sold_date_sk").isNotNull()
        & F.col("ss_item_sk").isNotNull()
        & F.col("ss_customer_sk").isNotNull()
        & (F.col("ss_quantity") > 0)
    )
    sr_agg = (
        spark.read.parquet(f"{bronze_root}/store_returns/")
        .filter(F.col("sr_item_sk").isNotNull() & F.col("sr_ticket_number").isNotNull())
        .groupBy("sr_item_sk", "sr_ticket_number")
        .agg(
            F.sum(F.coalesce(F.col("sr_return_quantity"), F.lit(0))).cast("long").alias("return_qty"),
            F.sum(F.coalesce(F.col("sr_return_amt"), F.lit(0.0))).cast("double").alias("return_amt"),
            F.sum(F.coalesce(F.col("sr_net_loss"), F.lit(0.0))).cast("double").alias("return_net_loss"),
        )
    )
    store_silver = (
        ss.join(
            sr_agg,
            (ss["ss_item_sk"] == sr_agg["sr_item_sk"])
            & (ss["ss_ticket_number"] == sr_agg["sr_ticket_number"]),
            how="left",
        )
        .select(
            F.lit("STORE").alias("channel"),
            ss["ss_sold_date_sk"].cast("long").alias("sold_date_sk"),
            ss["ss_item_sk"].cast("long").alias("item_sk"),
            ss["ss_customer_sk"].cast("long").alias("customer_sk"),
            ss["ss_cdemo_sk"].cast("long").alias("cdemo_sk"),
            ss["ss_ticket_number"].cast("long").alias("order_number"),
            ss["ss_quantity"].cast("long").alias("quantity"),
            F.coalesce(ss["ss_ext_sales_price"], F.lit(0.0)).cast("double").alias("gross_sales_price"),
            F.coalesce(ss["ss_ext_discount_amt"], F.lit(0.0)).cast("double").alias("discount_amt"),
            F.coalesce(ss["ss_net_paid"], F.lit(0.0)).cast("double").alias("net_paid"),
            F.coalesce(ss["ss_net_profit"], F.lit(0.0)).cast("double").alias("net_profit"),
            F.coalesce(sr_agg["return_qty"], F.lit(0)).cast("long").alias("return_qty"),
            F.coalesce(sr_agg["return_amt"], F.lit(0.0)).cast("double").alias("return_amt"),
            (
                F.coalesce(ss["ss_net_paid"], F.lit(0.0))
                - F.coalesce(sr_agg["return_amt"], F.lit(0.0))
            )
            .cast("double")
            .alias("net_paid_after_return"),
            (
                F.coalesce(ss["ss_net_profit"], F.lit(0.0))
                - F.coalesce(sr_agg["return_net_loss"], F.lit(0.0))
            )
            .cast("double")
            .alias("net_profit_after_return"),
        )
    )

    ws = spark.read.parquet(f"{bronze_root}/web_sales/").filter(
        F.col("ws_sold_date_sk").isNotNull()
        & F.col("ws_item_sk").isNotNull()
        & F.col("ws_bill_customer_sk").isNotNull()
        & (F.col("ws_quantity") > 0)
    )
    wr_agg = (
        spark.read.parquet(f"{bronze_root}/web_returns/")
        .filter(F.col("wr_item_sk").isNotNull() & F.col("wr_order_number").isNotNull())
        .groupBy("wr_item_sk", "wr_order_number")
        .agg(
            F.sum(F.coalesce(F.col("wr_return_quantity"), F.lit(0))).cast("long").alias("return_qty"),
            F.sum(F.coalesce(F.col("wr_return_amt"), F.lit(0.0))).cast("double").alias("return_amt"),
            F.sum(F.coalesce(F.col("wr_net_loss"), F.lit(0.0))).cast("double").alias("return_net_loss"),
        )
    )
    web_silver = (
        ws.join(
            wr_agg,
            (ws["ws_item_sk"] == wr_agg["wr_item_sk"])
            & (ws["ws_order_number"] == wr_agg["wr_order_number"]),
            how="left",
        )
        .select(
            F.lit("WEB").alias("channel"),
            ws["ws_sold_date_sk"].cast("long").alias("sold_date_sk"),
            ws["ws_item_sk"].cast("long").alias("item_sk"),
            ws["ws_bill_customer_sk"].cast("long").alias("customer_sk"),
            ws["ws_bill_cdemo_sk"].cast("long").alias("cdemo_sk"),
            ws["ws_order_number"].cast("long").alias("order_number"),
            ws["ws_quantity"].cast("long").alias("quantity"),
            F.coalesce(ws["ws_ext_sales_price"], F.lit(0.0)).cast("double").alias("gross_sales_price"),
            F.coalesce(ws["ws_ext_discount_amt"], F.lit(0.0)).cast("double").alias("discount_amt"),
            F.coalesce(ws["ws_net_paid"], F.lit(0.0)).cast("double").alias("net_paid"),
            F.coalesce(ws["ws_net_profit"], F.lit(0.0)).cast("double").alias("net_profit"),
            F.coalesce(wr_agg["return_qty"], F.lit(0)).cast("long").alias("return_qty"),
            F.coalesce(wr_agg["return_amt"], F.lit(0.0)).cast("double").alias("return_amt"),
            (
                F.coalesce(ws["ws_net_paid"], F.lit(0.0))
                - F.coalesce(wr_agg["return_amt"], F.lit(0.0))
            )
            .cast("double")
            .alias("net_paid_after_return"),
            (
                F.coalesce(ws["ws_net_profit"], F.lit(0.0))
                - F.coalesce(wr_agg["return_net_loss"], F.lit(0.0))
            )
            .cast("double")
            .alias("net_profit_after_return"),
        )
    )
    return store_silver.unionByName(web_silver)


def build_stage2a_vectorized(spark: SparkSession, silver_path: str) -> DataFrame:
    df = spark.read.parquet(silver_path)
    tier_expr = (
        F.when(
            (F.col("return_amt") > F.lit(0.5) * F.col("gross_sales_price"))
            & (F.col("return_qty") > F.lit(0)),
            F.lit("HIGH_RETURN_RISK"),
        )
        .when(
            (F.col("net_profit_after_return") > F.lit(50.0))
            & (F.col("discount_amt") == F.lit(0.0)),
            F.lit("VIP_FULL_PRICE"),
        )
        .when(
            F.col("discount_amt") > F.lit(0.25) * F.col("gross_sales_price"),
            F.lit("PROMO_DRIVEN"),
        )
        .otherwise(F.lit("STANDARD"))
    )
    margin_expr = F.round(
        F.col("net_profit_after_return")
        - (F.lit(0.15) * F.col("return_amt"))
        + (F.lit(0.05) * F.col("net_paid_after_return")),
        4,
    )
    return df.withColumn("order_risk_tier", tier_expr).withColumn("adjusted_margin_score", margin_expr)


def build_stage2b_udf(
    spark: SparkSession, silver_path: str, scale_preset: str
) -> tuple[DataFrame, float]:
    df = spark.read.parquet(silver_path)
    sample_fraction = 1.0
    if scale_preset in ("medium_100g", "large_500g"):
        df = df.filter(F.pmod(F.abs(F.col("order_number")), F.lit(20)) == F.lit(0))
        sample_fraction = 0.05

    @F.udf(returnType=StringType())
    def udf_risk_tier(return_amt, gross_sales_price, return_qty, net_profit_after_return, discount_amt):
        r_amt = float(return_amt or 0.0)
        g_price = float(gross_sales_price or 0.0)
        r_qty = int(return_qty or 0)
        np_after = float(net_profit_after_return or 0.0)
        disc = float(discount_amt or 0.0)
        if r_amt > 0.5 * g_price and r_qty > 0:
            return "HIGH_RETURN_RISK"
        if np_after > 50.0 and disc == 0.0:
            return "VIP_FULL_PRICE"
        if disc > 0.25 * g_price:
            return "PROMO_DRIVEN"
        return "STANDARD"

    @F.udf(returnType=DoubleType())
    def udf_margin_score(net_profit_after_return, return_amt, net_paid_after_return):
        np_after = float(net_profit_after_return or 0.0)
        r_amt = float(return_amt or 0.0)
        paid_after = float(net_paid_after_return or 0.0)
        return round(np_after - (0.15 * r_amt) + (0.05 * paid_after), 4)

    out_df = df.withColumn(
        "order_risk_tier",
        udf_risk_tier(
            F.col("return_amt"),
            F.col("gross_sales_price"),
            F.col("return_qty"),
            F.col("net_profit_after_return"),
            F.col("discount_amt"),
        ),
    ).withColumn(
        "adjusted_margin_score",
        udf_margin_score(
            F.col("net_profit_after_return"),
            F.col("return_amt"),
            F.col("net_paid_after_return"),
        ),
    )
    return out_df, sample_fraction


def build_stage3_gold(spark: SparkSession, enriched_path: str, bronze_root: str) -> DataFrame:
    s = spark.read.parquet(enriched_path)
    c = spark.read.parquet(f"{bronze_root}/customer/")
    cd = spark.read.parquet(f"{bronze_root}/customer_demographics/")
    i = spark.read.parquet(f"{bronze_root}/item/")
    d = spark.read.parquet(f"{bronze_root}/date_dim/")

    joined = (
        s.join(c, s["customer_sk"] == c["c_customer_sk"], how="inner")
        .join(
            cd,
            F.coalesce(s["cdemo_sk"], c["c_current_cdemo_sk"]) == cd["cd_demo_sk"],
            how="inner",
        )
        .join(i, s["item_sk"] == i["i_item_sk"], how="inner")
        .join(d, s["sold_date_sk"] == d["d_date_sk"], how="inner")
        .filter(
            i["i_category"].isNotNull()
            & cd["cd_credit_rating"].isNotNull()
            & d["d_date"].isNotNull()
        )
    )

    daily = joined.groupBy(
        d["d_date_sk"].alias("d_date_sk"),
        d["d_date"].alias("d_date"),
        d["d_year"].alias("d_year"),
        d["d_moy"].alias("d_moy"),
        s["channel"].alias("channel"),
        i["i_category"].alias("i_category"),
        cd["cd_credit_rating"].alias("cd_credit_rating"),
        s["order_risk_tier"].alias("order_risk_tier"),
    ).agg(
        F.count(F.lit(1)).alias("order_lines"),
        F.countDistinct(s["customer_sk"]).alias("unique_customers"),
        F.sum(s["gross_sales_price"]).alias("daily_gross_sales"),
        F.sum(s["return_amt"]).alias("daily_return_amt"),
        F.sum(s["net_paid_after_return"]).alias("daily_net_paid"),
        F.sum(s["net_profit_after_return"]).alias("daily_net_profit"),
        F.avg(s["adjusted_margin_score"]).alias("avg_margin_score"),
    )

    part_cols = ("channel", "i_category", "cd_credit_rating", "order_risk_tier")
    w_7d = Window.partitionBy(*part_cols).orderBy(F.col("d_date_sk").asc()).rowsBetween(-6, 0)
    w_30d = Window.partitionBy(*part_cols).orderBy(F.col("d_date_sk").asc()).rowsBetween(-29, 0)
    w_cat = Window.partitionBy("d_date_sk", "channel")

    denom_30d = F.sum("daily_gross_sales").over(w_30d)
    denom_cat = F.sum("daily_net_paid").over(w_cat)

    return (
        daily.withColumn("rolling_7d_net_paid", F.sum("daily_net_paid").over(w_7d))
        .withColumn("rolling_30d_net_paid", F.sum("daily_net_paid").over(w_30d))
        .withColumn(
            "rolling_30d_return_rate",
            F.when(denom_30d != F.lit(0.0), F.sum("daily_return_amt").over(w_30d) / denom_30d)
            .otherwise(F.lit(None).cast("double")),
        )
        .withColumn(
            "category_daily_share",
            F.when(denom_cat != F.lit(0.0), F.col("daily_net_paid") / denom_cat)
            .otherwise(F.lit(None).cast("double")),
        )
    )


def write_json_gcs(spark: SparkSession, gcs_uri: str, payload: dict[str, Any]) -> None:
    jvm = spark._jvm
    conf = spark._jsc.hadoopConfiguration()
    uri = jvm.java.net.URI(gcs_uri)
    fs = jvm.org.apache.hadoop.fs.FileSystem.get(uri, conf)
    path = jvm.org.apache.hadoop.fs.Path(gcs_uri)
    out = fs.create(path, True)
    writer = jvm.java.io.BufferedWriter(jvm.java.io.OutputStreamWriter(out, "UTF-8"))
    try:
        writer.write(json.dumps(payload, indent=2))
        writer.flush()
    finally:
        writer.close()


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv)
    bucket = args.bench_bucket.removeprefix("gs://").rstrip("/")
    run_udf = args.run_python_udf.lower() in {"1", "true", "yes"}
    bronze_root = f"gs://{bucket}/bronze/{args.scale_preset}"
    silver_root = f"gs://{bucket}/silver/{args.scale_preset}/{args.engine_variant}"
    gold_root = f"gs://{bucket}/gold/{args.scale_preset}/{args.engine_variant}"

    spark = SparkSession.builder.appName(
        f"lakehouse-bench-{args.engine_variant}-{args.scale_preset}"
    ).getOrCreate()
    try:
        runtime_conf = spark.conf.get("spark.dataproc.lightningEngine.runtime", "default")
        nqe_enabled = args.engine_variant == "spark_lightning_nqe" or runtime_conf == "native"
        # Warm up YARN executors prior to timing Stage 1.
        spark.range(1).count()

        records: list[dict[str, Any]] = []
        plans: dict[str, str] = {}

        def _record_stage(
            stage_id: str,
            df: DataFrame,
            out_path: str,
            checksum_col: str,
            sample_fraction: float = 1.0,
            fallback_note: str = "",
        ) -> None:
            t0 = time.time()
            df.write.mode("overwrite").parquet(out_path)
            t1 = time.time()
            plans[stage_id] = capture_formatted_plan(spark, df)
            note = fallback_note
            if stage_id == "stage2b_silver_udf" and nqe_enabled:
                markers = [
                    tok
                    for tok in ("VeloxColumnarToRow", "BatchEvalPython", "ArrowEvalPython", "RowToVeloxColumnar")
                    if tok in plans[stage_id]
                ]
                if markers:
                    note = f"NQE columnar-to-row Python UDF fallback detected: {', '.join(markers)}"
                else:
                    note = "Python UDF executed via JVM/Python boundary outside native C++ codegen"
            cnt, chk = measure_stage_output(spark, out_path, checksum_col)
            wall = round(t1 - t0, 4)
            records.append(
                {
                    "run_id": args.run_id,
                    "scale_preset": args.scale_preset,
                    "engine_variant": args.engine_variant,
                    "storage_backend": "gcs_parquet",
                    "stage_id": stage_id,
                    "start_epoch_sec": round(t0, 4),
                    "end_epoch_sec": round(t1, 4),
                    "wall_clock_sec": wall,
                    "output_row_count": cnt,
                    "checksum_net_paid": chk,
                    "worker_machine_type": args.worker_machine_type,
                    "total_worker_vcpus": int(args.total_worker_vcpus),
                    "vcpu_seconds": round(wall * int(args.total_worker_vcpus), 4),
                    "bq_slot_cap": None,
                    "bq_total_slot_ms": None,
                    "bq_bytes_processed": None,
                    "nqe_enabled": bool(nqe_enabled),
                    "nqe_fallback_note": note,
                    "sample_fraction": sample_fraction,
                    "spark_lightning_runtime_conf": runtime_conf,
                }
            )

        stage1_out = f"{silver_root}/silver_omnichannel_sales/"
        _record_stage(
            "stage1_bronze_to_silver",
            build_stage1_silver(spark, bronze_root),
            stage1_out,
            "net_paid_after_return",
        )

        stage2a_out = f"{silver_root}/silver_enriched_sales/"
        _record_stage(
            "stage2a_silver_vectorized",
            build_stage2a_vectorized(spark, stage1_out),
            stage2a_out,
            "net_paid_after_return",
        )

        if run_udf:
            stage2b_out = f"{silver_root}/silver_udf_benchmark_out/"
            df_2b, sample_frac = build_stage2b_udf(spark, stage1_out, args.scale_preset)
            _record_stage(
                "stage2b_silver_udf",
                df_2b,
                stage2b_out,
                "net_paid_after_return",
                sample_fraction=sample_frac,
            )

        stage3_out = f"{gold_root}/gold_omnichannel_kpi_mart/"
        _record_stage(
            "stage3_silver_to_gold",
            build_stage3_gold(spark, stage2a_out, bronze_root),
            stage3_out,
            "daily_net_paid",
        )

        payload = {
            "run_id": args.run_id,
            "engine_variant": args.engine_variant,
            "scale_preset": args.scale_preset,
            "spark_lightning_runtime_conf": runtime_conf,
            "nqe_enabled": bool(nqe_enabled),
            "records": records,
            "plans": plans,
        }
        write_json_gcs(spark, args.metrics_gcs_uri, payload)
        print("BENCHMARK_METRICS_JSON_BEGIN")
        print(json.dumps(payload))
        print("BENCHMARK_METRICS_JSON_END")
    finally:
        spark.stop()
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''


def resolve_runner_script(temp_dir: str | None = None) -> tuple[Path, bool]:
    '''Use shared scripts/run_spark_medallion.py when present; else materialize temp fallback.'''
    shared_path = REPO_ROOT / "scripts" / "run_spark_medallion.py"
    if shared_path.is_file():
        return shared_path, False
    if temp_dir is None:
        raise ValueError("temp_dir must be provided when generating fallback runner.")
    fallback_path = Path(temp_dir) / "run_spark_medallion_fallback.py"
    fallback_path.write_text(FALLBACK_SPARK_RUNNER_CODE, encoding="utf-8")
    return fallback_path, True


def read_gcs_json(gcs_uri: str) -> dict[str, Any]:
    '''Download and parse a JSON object from gs://bucket/path.'''
    from google.cloud import storage

    if not gcs_uri.startswith("gs://"):
        raise ValueError(f"Expected gs:// URI, got '{gcs_uri}'.")
    bucket_name, blob_path = gcs_uri.removeprefix("gs://").split("/", 1)
    client = storage.Client(project=PROJECT_ID)
    raw = client.bucket(bucket_name).blob(blob_path).download_as_text(encoding="utf-8")
    return json.loads(raw)


def _extract_stdout_payload(stdout: str) -> dict[str, Any] | None:
    begin = "BENCHMARK_METRICS_JSON_BEGIN"
    end = "BENCHMARK_METRICS_JSON_END"
    if begin in stdout and end in stdout:
        body = stdout.split(begin, 1)[1].split(end, 1)[0].strip()
        return json.loads(body)
    return None


def submit_lightning_pass(engine_variant: str, runtime_mode: str) -> dict[str, Any]:
    '''Submit a single Lightning Engine Medallion pass and return parsed telemetry + plans.'''
    metrics_gcs_uri = f"gs://{BENCH_BUCKET}/metrics/{RUN_ID}/{engine_variant}.json"
    cli_args = [
        "--engine-variant",
        engine_variant,
        "--scale-preset",
        SCALE_PRESET,
        "--bench-bucket",
        BENCH_BUCKET,
        "--run-id",
        RUN_ID,
        "--worker-machine-type",
        WORKER_MACHINE_TYPE,
        "--total-worker-vcpus",
        str(TOTAL_WORKER_VCPUS),
        "--run-python-udf",
        "true" if RUN_PYTHON_UDF_STAGE else "false",
        "--udf-max-sample-rows",
        str(UDF_MAX_SAMPLE_ROWS),
        "--metrics-gcs-uri",
        metrics_gcs_uri,
    ]

    ctx = (
        nullcontext(None)
        if (REPO_ROOT / "scripts" / "run_spark_medallion.py").is_file()
        else tempfile.TemporaryDirectory()
    )
    with ctx as tmp:
        runner_path, is_fallback = resolve_runner_script(tmp)
        print(
            f"Submitting {engine_variant} (runtime={runtime_mode}) using "
            f"{'fallback runner' if is_fallback else runner_path.relative_to(REPO_ROOT)} -> {metrics_gcs_uri}"
        )

        if EXECUTION_MODE == "local_spark":
            # Direct execution when notebook kernel runs inside the Dataproc cluster itself.
            cmd = [sys.executable, str(runner_path), *cli_args]
        else:
            # NOTE: `gcloud dataproc jobs submit pyspark --properties` takes KEY=VALUE WITHOUT `spark:` prefix.
            cmd = [
                "gcloud",
                "dataproc",
                "jobs",
                "submit",
                "pyspark",
                str(runner_path),
                f"--project={PROJECT_ID}",
                f"--region={DATAPROC_REGION}",
                f"--cluster={CLUSTER_LGT_NAME}",
                f"--properties=spark.dataproc.lightningEngine.runtime={runtime_mode}",
                "--",
                *cli_args,
            ]

        proc = subprocess.run(cmd, text=True, capture_output=True)
        if proc.returncode != 0:
            print(proc.stdout[-4000:])
            print(proc.stderr[-4000:], file=sys.stderr)
            raise RuntimeError(f"Dataproc PySpark job failed for {engine_variant} (exit={proc.returncode}).")

        try:
            payload = read_gcs_json(metrics_gcs_uri)
        except Exception as gcs_err:
            payload = _extract_stdout_payload(proc.stdout)
            if payload is None:
                raise RuntimeError(f"Failed to read {metrics_gcs_uri} and no stdout fallback found.") from gcs_err

        payload.setdefault("metrics_gcs_uri", metrics_gcs_uri)
        return payload

## 4. Pass 1 — Base Lightning Engine (`spark_lightning_base`, `runtime=default`)

Submits `run_spark_medallion.py` to `${CLUSTER_LGT_NAME}` with:
`--properties="spark.dataproc.lightningEngine.runtime=default"` (`nqe_enabled = False`).

In [ ]:
base_payload = submit_lightning_pass(
    engine_variant="spark_lightning_base",
    runtime_mode="default",
)
base_records = base_payload["records"]
base_plans = base_payload.get("plans", {})

print(
    f"Pass 1 complete: engine_variant=spark_lightning_base | "
    f"runtime_conf={base_payload.get('spark_lightning_runtime_conf', 'default')} | "
    f"stages={len(base_records)}"
)
base_df_preview = pd.DataFrame(base_records)
base_cols = [
    c
    for c in ("stage_id", "wall_clock_sec", "vcpu_seconds", "output_row_count", "checksum_net_paid", "nqe_enabled")
    if c in base_df_preview.columns
]
display(base_df_preview[base_cols])

## 5. Pass 2 — Lightning Engine + Gluten/Velox C++ NQE (`spark_lightning_nqe`, `runtime=native`)

Submits the exact same runner to `${CLUSTER_LGT_NAME}` with:
`--properties="spark.dataproc.lightningEngine.runtime=native"` (`nqe_enabled = True`).

In [ ]:
nqe_payload = submit_lightning_pass(
    engine_variant="spark_lightning_nqe",
    runtime_mode="native",
)
nqe_records = nqe_payload["records"]
nqe_plans = nqe_payload.get("plans", {})

print(
    f"Pass 2 complete: engine_variant=spark_lightning_nqe | "
    f"runtime_conf={nqe_payload.get('spark_lightning_runtime_conf', 'native')} | "
    f"stages={len(nqe_records)}"
)
nqe_df_preview = pd.DataFrame(nqe_records)
nqe_cols = [
    c
    for c in (
        "stage_id",
        "wall_clock_sec",
        "vcpu_seconds",
        "output_row_count",
        "checksum_net_paid",
        "nqe_enabled",
        "nqe_fallback_note",
    )
    if c in nqe_df_preview.columns
]
display(nqe_df_preview[nqe_cols])

## 6. Persist Unified Telemetry (`results/live_runs/benchmark_runs.jsonl` & BigQuery)

Persists all stage records for both `spark_lightning_base` (`nqe_enabled=False`) and `spark_lightning_nqe` (`nqe_enabled=True`) via `scripts.metrics_helper.log_stage_metric(...)` (using a deterministic schema-matched fallback only if `metrics_helper.py` is absent).

In [ ]:
def _fallback_log_stage_metric(
    *,
    run_id: str,
    scale_preset: str,
    engine_variant: str,
    storage_backend: str,
    stage_id: str,
    start_epoch_sec: float,
    end_epoch_sec: float,
    output_row_count: int,
    checksum_net_paid: float | None = None,
    worker_machine_type: str | None = None,
    total_worker_vcpus: int | None = None,
    bq_slot_cap: int | None = None,
    bq_total_slot_ms: int | None = None,
    bq_bytes_processed: int | None = None,
    nqe_enabled: bool = False,
    nqe_fallback_note: str = "",
    est_list_cost_usd: float | None = None,
    sample_fraction: float = 1.0,
    jsonl_path: str | Path | None = None,
    write_to_bq: bool = False,
    project_id: str | None = None,
    ds_metrics: str | None = None,
) -> dict[str, Any]:
    '''Schema-identical SPEC.md §4 telemetry fallback when scripts/metrics_helper.py is absent.'''
    wall = round(float(end_epoch_sec) - float(start_epoch_sec), 4)
    vcpu_sec = round(wall * int(total_worker_vcpus), 4) if total_worker_vcpus else None
    if est_list_cost_usd is None and vcpu_sec is not None:
        # Parameterized default us-central1 list estimate (n2d-standard + Dataproc fee + Local SSD + Lightning surcharge)
        hourly_per_vcpu = 0.04225 + 0.010 + 0.00493 + (0.0025 if "lightning" in engine_variant else 0.0)
        est_list_cost_usd = round((vcpu_sec / 3600.0) * hourly_per_vcpu, 6)
    rec = {
        "run_id": run_id,
        "timestamp_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "scale_preset": scale_preset,
        "engine_variant": engine_variant,
        "storage_backend": storage_backend,
        "stage_id": stage_id,
        "start_epoch_sec": round(float(start_epoch_sec), 4),
        "end_epoch_sec": round(float(end_epoch_sec), 4),
        "wall_clock_sec": wall,
        "output_row_count": int(output_row_count),
        "checksum_net_paid": checksum_net_paid,
        "worker_machine_type": worker_machine_type,
        "total_worker_vcpus": total_worker_vcpus,
        "vcpu_seconds": vcpu_sec,
        "bq_slot_cap": bq_slot_cap,
        "bq_total_slot_ms": bq_total_slot_ms,
        "bq_bytes_processed": bq_bytes_processed,
        "nqe_enabled": bool(nqe_enabled),
        "nqe_fallback_note": nqe_fallback_note,
        "est_list_cost_usd": est_list_cost_usd,
        "sample_fraction": float(sample_fraction),
    }
    target = Path(jsonl_path) if jsonl_path else LIVE_JSONL_PATH
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(rec) + "\n")
    return rec


_logger_fn = log_stage_metric if METRICS_HELPER_AVAILABLE else _fallback_log_stage_metric
_logger_params = set(inspect.signature(_logger_fn).parameters.keys())

logged_rows: list[dict[str, Any]] = []
for expected_nqe, raw_records in ((False, base_records), (True, nqe_records)):
    for rec in raw_records:
        kwargs: dict[str, Any] = {
            "run_id": rec.get("run_id", RUN_ID),
            "scale_preset": rec.get("scale_preset", SCALE_PRESET),
            "engine_variant": rec["engine_variant"],
            "storage_backend": rec.get("storage_backend", "gcs_parquet"),
            "stage_id": rec["stage_id"],
            "start_epoch_sec": float(rec["start_epoch_sec"]),
            "end_epoch_sec": float(rec["end_epoch_sec"]),
            "output_row_count": int(rec["output_row_count"]),
            "checksum_net_paid": rec.get("checksum_net_paid"),
            "worker_machine_type": rec.get("worker_machine_type", WORKER_MACHINE_TYPE),
            "total_worker_vcpus": int(rec.get("total_worker_vcpus", TOTAL_WORKER_VCPUS)),
            "bq_slot_cap": rec.get("bq_slot_cap"),
            "bq_total_slot_ms": rec.get("bq_total_slot_ms"),
            "bq_bytes_processed": rec.get("bq_bytes_processed"),
            "nqe_enabled": expected_nqe,
            "nqe_fallback_note": str(rec.get("nqe_fallback_note", "")),
            "est_list_cost_usd": rec.get("est_list_cost_usd"),
            "jsonl_path": LIVE_JSONL_PATH,
            "write_to_bq": WRITE_METRICS_TO_BQ,
            "project_id": PROJECT_ID,
            "ds_metrics": DS_METRICS,
        }
        if "sample_fraction" in _logger_params:
            kwargs["sample_fraction"] = float(rec.get("sample_fraction", 1.0))
        logged_rows.append(_logger_fn(**kwargs))

logged_df = pd.DataFrame(logged_rows)
_display_jsonl = LIVE_JSONL_PATH.relative_to(REPO_ROOT) if LIVE_JSONL_PATH.is_relative_to(REPO_ROOT) else LIVE_JSONL_PATH
print(
    f"Persisted {len(logged_df)} stage records to {_display_jsonl} "
    f"(write_to_bq={WRITE_METRICS_TO_BQ})."
)
display(
    logged_df[
        [
            "engine_variant",
            "stage_id",
            "wall_clock_sec",
            "vcpu_seconds",
            "output_row_count",
            "checksum_net_paid",
            "nqe_enabled",
            "est_list_cost_usd",
        ]
    ]
)

## 7. Internal Correctness & Checksum Parity (`spark_lightning_base` vs. `spark_lightning_nqe`)

Validates `SPEC.md` §3 parity invariants between Base Lightning and Gluten/Velox C++ NQE:
- `output_row_count` **MUST** match 100% across every stage.
- `checksum_net_paid` **MUST** agree within `±0.01%` (`0.0001` relative tolerance).

In [ ]:
CHECKSUM_REL_TOLERANCE = 0.0001  # +/- 0.01%


def evaluate_internal_parity(
    base_recs: list[dict[str, Any]],
    nqe_recs: list[dict[str, Any]],
    rel_tol: float = CHECKSUM_REL_TOLERANCE,
) -> pd.DataFrame:
    '''Compare Base vs NQE row counts and checksum_net_paid across all executed stages.'''
    base_by_stage = {r["stage_id"]: r for r in base_recs}
    nqe_by_stage = {r["stage_id"]: r for r in nqe_recs}
    if set(base_by_stage) != set(nqe_by_stage):
        raise AssertionError(
            f"Stage mismatch between Base {sorted(base_by_stage)} and NQE {sorted(nqe_by_stage)}."
        )

    rows: list[dict[str, Any]] = []
    for stage_id in base_by_stage:
        b = base_by_stage[stage_id]
        n = nqe_by_stage[stage_id]
        b_cnt = int(b["output_row_count"])
        n_cnt = int(n["output_row_count"])
        row_match = b_cnt == n_cnt

        b_chk = float(b["checksum_net_paid"] or 0.0)
        n_chk = float(n["checksum_net_paid"] or 0.0)
        rel_diff = abs(n_chk - b_chk) / max(abs(b_chk), 1e-9)
        chk_match = rel_diff <= rel_tol

        rows.append(
            {
                "stage_id": stage_id,
                "base_row_count": b_cnt,
                "nqe_row_count": n_cnt,
                "row_delta": n_cnt - b_cnt,
                "base_checksum": round(b_chk, 2),
                "nqe_checksum": round(n_chk, 2),
                "checksum_rel_diff_pct": round(rel_diff * 100.0, 6),
                "parity_status": "PASS" if (row_match and chk_match) else "FAIL",
            }
        )
    parity_df = pd.DataFrame(rows)
    if not (parity_df["parity_status"] == "PASS").all():
        failing = parity_df[parity_df["parity_status"] != "PASS"]
        raise AssertionError(f"Parity check FAILED between Base and NQE:\n{failing.to_string(index=False)}")
    return parity_df


parity_df = evaluate_internal_parity(base_records, nqe_records)
display(parity_df)

## 8. Physical Plan Inspection: Gluten/Velox C++ NQE vs. `@F.udf` Fallback

Parses the `plans` dictionary (`df.explain("formatted")`) returned in `spark_lightning_base.json` and `spark_lightning_nqe.json`:
- **Relational Stages (`1`, `2A`, `3`) on `runtime=native`:** Look for native C++ vectorized operators (`NativeFileScan`, `BatchScanExecTransformer`, `FileSourceScanExecTransformer`, `FilterExecTransformer`, `ProjectExecTransformer`, `HashAggregateTransformer`, `ShuffledHashJoinExecTransformer`, `BroadcastHashJoinExecTransformer`, `WindowExecTransformer`, `WholeStageCodegenTransformer`).
- **Stage `2B` (`@F.udf`):** Look for the explicit columnar-to-row boundary (`VeloxColumnarToRow -> BatchEvalPythonExec / ArrowEvalPythonExec -> RowToVeloxColumnar`), demonstrating why scalar Python UDFs force execution out of Velox C++ off-heap batches into JVM/Python row serialization.

In [ ]:
NATIVE_VELOX_TOKENS = (
    "NativeFileScan",
    "BatchScanExecTransformer",
    "FileSourceScanExecTransformer",
    "FilterExecTransformer",
    "ProjectExecTransformer",
    "FlushableHashAggregateTransformer",
    "HashAggregateTransformer",
    "SortAggregateExecTransformer",
    "ShuffledHashJoinExecTransformer",
    "BroadcastHashJoinExecTransformer",
    "SortMergeJoinExecTransformer",
    "WindowExecTransformer",
    "SortExecTransformer",
    "ExpandExecTransformer",
    "TakeOrderedAndProjectExecTransformer",
    "ColumnarShuffleExchangeExec",
    "ColumnarBroadcastExchange",
    "ColumnarExchange",
    "ColumnarUnionExec",
    "InputIteratorTransformer",
    "VeloxResizeBatches",
    "WholeStageCodegenTransformer",
    "ColumnarWriteFilesExec",
)
TRANSITION_TOKENS = (
    "VeloxColumnarToRowExec",
    "VeloxColumnarToRow",
    "RowToVeloxColumnarExec",
    "RowToVeloxColumnar",
    "ColumnarToRow",
    "RowToColumnar",
)
PYTHON_JVM_FALLBACK_TOKENS = (
    "BatchEvalPythonExec",
    "BatchEvalPython",
    "ArrowEvalPythonExec",
    "ArrowEvalPython",
    "WholeStageCodegen",
    "FileScan parquet",
    "Scan parquet",
    "Filter",
    "Project",
    "HashAggregate",
    "BroadcastHashJoin",
    "ShuffledHashJoin",
    "SortMergeJoin",
    "Window",
)


def extract_plan_tree_lines(plan_text: str) -> list[str]:
    '''Extract the top physical plan tree block before node detail sections.'''
    tree_lines: list[str] = []
    for line in plan_text.splitlines():
        stripped = line.strip()
        if re.match(r"^\(\d+\)\s+", stripped):
            break
        if stripped:
            tree_lines.append(line)
    return tree_lines


def parse_plan_operators(plan_text: str) -> dict[str, Any]:
    '''Count native Velox transformers, columnar-row transitions, and Python/JVM operators.'''
    if not plan_text:
        return {
            "native_ops": {},
            "transition_ops": {},
            "python_jvm_ops": {},
            "native_count": 0,
            "transition_count": 0,
            "python_eval_count": 0,
            "offload_status": "PLAN_MISSING",
        }

    # Parse only the top tree lines (falling back to `(N) Operator` headers if no tree exists)
    # so each operator node is counted once and column names inside details never cause false hits.
    tree_lines = extract_plan_tree_lines(plan_text)
    candidate_lines: list[str] = []
    codegen_stages: set[str] = set()
    for line in tree_lines:
        cleaned = re.sub(r"^[\s|:\+\-]+", "", line)
        cg_match = re.match(r"^\*\((\d+)\)\s*(.*)$", cleaned)
        if cg_match:
            codegen_stages.add(cg_match.group(1))
            cleaned = cg_match.group(2)
        if cleaned and not cleaned.startswith("== ") and not cleaned.startswith("AdaptiveSparkPlan"):
            candidate_lines.append(cleaned)

    if not candidate_lines:
        for line in plan_text.splitlines():
            m = re.match(r"^\(\d+\)\s+(.+)$", line.strip())
            if m:
                candidate_lines.append(m.group(1).strip())

    def _count_tokens(tokens: tuple[str, ...]) -> dict[str, int]:
        counts: dict[str, int] = {}
        for cand in candidate_lines:
            head = cand.split("[", 1)[0].split("(", 1)[0].strip()
            for tok in tokens:
                if cand.startswith(tok) or re.search(rf"\b{re.escape(tok)}\b", head):
                    counts[tok] = counts.get(tok, 0) + 1
                    break
        return counts

    native_ops = _count_tokens(NATIVE_VELOX_TOKENS)
    transition_ops = _count_tokens(TRANSITION_TOKENS)
    python_jvm_ops = _count_tokens(PYTHON_JVM_FALLBACK_TOKENS)
    if codegen_stages and "WholeStageCodegen" not in python_jvm_ops:
        python_jvm_ops["WholeStageCodegen"] = len(codegen_stages)

    native_count = sum(native_ops.values())
    transition_count = sum(transition_ops.values())
    python_eval_count = sum(
        cnt
        for tok, cnt in python_jvm_ops.items()
        if tok in {"BatchEvalPythonExec", "BatchEvalPython", "ArrowEvalPythonExec", "ArrowEvalPython"}
    )

    if python_eval_count > 0 and (native_count > 0 or transition_count > 0):
        offload_status = "PARTIAL_UDF_FALLBACK"
    elif native_count > 0:
        offload_status = "FULL_NATIVE_VELOX"
    elif python_eval_count > 0:
        offload_status = "JVM_PYTHON_UDF"
    else:
        offload_status = "JVM_WHOLE_STAGE_CODEGEN"

    return {
        "native_ops": native_ops,
        "transition_ops": transition_ops,
        "python_jvm_ops": python_jvm_ops,
        "native_count": native_count,
        "transition_count": transition_count,
        "python_eval_count": python_eval_count,
        "offload_status": offload_status,
    }


def _fmt_op_dict(d: dict[str, int]) -> str:
    return ", ".join(f"{k}({v})" for k, v in d.items()) if d else "none"


def build_plan_comparison_table(
    base_plan_map: dict[str, str], nqe_plan_map: dict[str, str]
) -> pd.DataFrame:
    '''Build side-by-side operator comparison table across all Medallion stages.'''
    stage_order = [
        "stage1_bronze_to_silver",
        "stage2a_silver_vectorized",
        "stage2b_silver_udf",
        "stage3_silver_to_gold",
    ]
    rows: list[dict[str, Any]] = []
    for stage_id in stage_order:
        if stage_id not in base_plan_map and stage_id not in nqe_plan_map:
            continue
        b_parsed = parse_plan_operators(base_plan_map.get(stage_id, ""))
        n_parsed = parse_plan_operators(nqe_plan_map.get(stage_id, ""))
        nqe_fallback_subset = {
            k: v
            for k, v in n_parsed["python_jvm_ops"].items()
            if "Python" in k or n_parsed["offload_status"] != "FULL_NATIVE_VELOX"
        }
        rows.append(
            {
                "stage_id": stage_id,
                "base_status": b_parsed["offload_status"],
                "base_jvm_python_ops": _fmt_op_dict(b_parsed["python_jvm_ops"]),
                "nqe_status": n_parsed["offload_status"],
                "nqe_native_velox_ops": _fmt_op_dict(n_parsed["native_ops"]),
                "nqe_transitions": _fmt_op_dict(n_parsed["transition_ops"]),
                "nqe_fallback_eval_ops": _fmt_op_dict(nqe_fallback_subset),
            }
        )
    return pd.DataFrame(rows)


plan_comparison_df = build_plan_comparison_table(base_plans, nqe_plans)
display(plan_comparison_df)

for highlight_stage in ("stage1_bronze_to_silver", "stage2b_silver_udf", "stage3_silver_to_gold"):
    if highlight_stage in nqe_plans:
        print(f"\n=== NQE Formatted Plan Tree Excerpt: {highlight_stage} ===")
        tree = extract_plan_tree_lines(nqe_plans[highlight_stage])
        print("\n".join(tree[:45]))
        if len(tree) > 45:
            print(f"... ({len(tree) - 45} additional tree lines truncated)")

## 9. Stage `2B` `@F.udf` Penalty, Stage Speedups & Lightning Surcharge Break-Even

Evaluates three commercial and technical questions:
1. **UDF Penalty (`stage2b_silver_udf` vs. `stage2a_silver_vectorized`):** Quantifies wall-clock and `vcpu_seconds` inflation normalized by `sample_fraction` (`1.0` on `small_1g`, `0.05` on `medium_100g`/`large_500g`).
2. **Stage-by-Stage Speedup:** Compares `spark_lightning_base` vs. `spark_lightning_nqe` (and includes `spark_standard` from `results/live_runs/benchmark_runs.jsonl` whenever present for the same `SCALE_PRESET`).
3. **Break-Even Surcharge Analysis (`+$0.0025/vCPU-hr`):** Computes required break-even speedup `%` across **On-Demand** vs. **`15% Primary (1-Yr CUD) + 85% Spot EFM`** vs. **`1-Yr CUD`** vs. **`3-Yr CUD`** (`SPEC.md` §4), comparing against actual measured relational speedup `%`.

In [ ]:
# Parameterized us-central1 GCE + Dataproc pricing defaults (USD per worker vCPU-hour; SPEC.md §4)
GCE_N2D_VCPU_HR_ONDEMAND = 0.04225  # n2d-standard (1 vCPU + 4 GiB RAM) list us-central1
GCE_N2D_VCPU_HR_1YR_CUD = 0.02662   # ~37% 1-Yr CUD discount
GCE_N2D_VCPU_HR_3YR_CUD = 0.01901   # ~55% 3-Yr CUD discount
GCE_N2D_VCPU_HR_SPOT = 0.01056      # Representative Spot (~75% discount)
DATAPROC_FEE_VCPU_HR = 0.01000
LOCAL_SSD_VCPU_HR = 0.00493         # 1x 375 GB NVMe Local SSD amortized across 8 worker vCPUs
LIGHTNING_SURCHARGE_VCPU_HR = 0.00250

STAGE_ORDER = [
    "stage1_bronze_to_silver",
    "stage2a_silver_vectorized",
    "stage2b_silver_udf",
    "stage3_silver_to_gold",
]


def compute_stage2b_penalty(records: list[dict[str, Any]]) -> pd.DataFrame | None:
    '''Compare Stage 2A vectorized vs Stage 2B @F.udf normalized for sample_fraction.'''
    by_eng_stage = {(r["engine_variant"], r["stage_id"]): r for r in records}
    rows: list[dict[str, Any]] = []
    for engine in ("spark_lightning_base", "spark_lightning_nqe"):
        r2a = by_eng_stage.get((engine, "stage2a_silver_vectorized"))
        r2b = by_eng_stage.get((engine, "stage2b_silver_udf"))
        if not r2a or not r2b:
            continue
        frac = float(r2b.get("sample_fraction", 1.0) or 1.0)
        w2a = float(r2a["wall_clock_sec"])
        w2b = float(r2b["wall_clock_sec"])
        v2a = float(r2a.get("vcpu_seconds") or (w2a * TOTAL_WORKER_VCPUS))
        v2b = float(r2b.get("vcpu_seconds") or (w2b * TOTAL_WORKER_VCPUS))

        norm_w2b = w2b / frac
        norm_v2b = v2b / frac
        rows.append(
            {
                "engine_variant": engine,
                "stage2a_wall_sec": round(w2a, 2),
                "stage2b_raw_wall_sec": round(w2b, 2),
                "stage2b_sample_fraction": frac,
                "stage2b_norm_full_wall_sec": round(norm_w2b, 2),
                "raw_slowdown_x": round(w2b / max(w2a, 1e-6), 2),
                "normalized_slowdown_x": round(norm_w2b / max(w2a, 1e-6), 2),
                "stage2a_vcpu_sec": round(v2a, 2),
                "stage2b_norm_vcpu_sec": round(norm_v2b, 2),
                "vcpu_sec_penalty_pct": round(((norm_v2b - v2a) / max(v2a, 1e-6)) * 100.0, 1),
                "nqe_fallback_note": r2b.get("nqe_fallback_note", ""),
            }
        )
    return pd.DataFrame(rows) if rows else None


def load_optional_standard_baseline() -> dict[str, dict[str, Any]]:
    '''Load latest spark_standard run for SCALE_PRESET from live JSONL if already recorded.'''
    if not LIVE_JSONL_PATH.is_file():
        return {}
    if METRICS_HELPER_AVAILABLE:
        loaded = load_benchmark_runs(live_path=LIVE_JSONL_PATH)
        if isinstance(loaded, tuple):
            loaded = loaded[0]
        df_all = loaded if isinstance(loaded, pd.DataFrame) else pd.DataFrame(loaded)
    else:
        with LIVE_JSONL_PATH.open("r", encoding="utf-8") as fh:
            df_all = pd.DataFrame([json.loads(line) for line in fh if line.strip()])
    if df_all.empty:
        return {}
    sub = df_all[(df_all["scale_preset"] == SCALE_PRESET) & (df_all["engine_variant"] == "spark_standard")].copy()
    if sub.empty:
        return {}
    sub = sub.sort_values("end_epoch_sec").drop_duplicates(subset=["stage_id"], keep="last")
    return {row["stage_id"]: row.to_dict() for _, row in sub.iterrows()}


def build_speedup_table(
    base_recs: list[dict[str, Any]],
    nqe_recs: list[dict[str, Any]],
    std_by_stage: dict[str, dict[str, Any]],
) -> pd.DataFrame:
    '''Build stage-by-stage speedup table comparing Base, NQE, and optional Standard.'''
    b_map = {r["stage_id"]: r for r in base_recs}
    n_map = {r["stage_id"]: r for r in nqe_recs}
    rows: list[dict[str, Any]] = []
    for stage_id in [s for s in STAGE_ORDER if s in b_map and s in n_map]:
        bw = float(b_map[stage_id]["wall_clock_sec"])
        nw = float(n_map[stage_id]["wall_clock_sec"])
        bv = float(b_map[stage_id].get("vcpu_seconds") or (bw * TOTAL_WORKER_VCPUS))
        nv = float(n_map[stage_id].get("vcpu_seconds") or (nw * TOTAL_WORKER_VCPUS))
        entry: dict[str, Any] = {
            "stage_id": stage_id,
            "base_wall_sec": round(bw, 2),
            "nqe_wall_sec": round(nw, 2),
            "nqe_vs_base_speedup_x": round(bw / max(nw, 1e-6), 2),
            "nqe_time_reduction_pct": round(((bw - nw) / max(bw, 1e-6)) * 100.0, 1),
            "base_vcpu_sec": round(bv, 2),
            "nqe_vcpu_sec": round(nv, 2),
        }
        if stage_id in std_by_stage:
            sw = float(std_by_stage[stage_id]["wall_clock_sec"])
            entry["std_wall_sec"] = round(sw, 2)
            entry["base_vs_std_speedup_x"] = round(sw / max(bw, 1e-6), 2)
            entry["nqe_vs_std_speedup_x"] = round(sw / max(nw, 1e-6), 2)
        rows.append(entry)

    # End-to-end relational subtotal (Stages 1 + 2A + 3) excluding Stage 2B UDF contrast.
    rel_stages = [s for s in ("stage1_bronze_to_silver", "stage2a_silver_vectorized", "stage3_silver_to_gold") if s in b_map]
    if rel_stages:
        rel_bw = sum(float(b_map[s]["wall_clock_sec"]) for s in rel_stages)
        rel_nw = sum(float(n_map[s]["wall_clock_sec"]) for s in rel_stages)
        rel_bv = sum(
            float(b_map[s].get("vcpu_seconds") or (float(b_map[s]["wall_clock_sec"]) * TOTAL_WORKER_VCPUS))
            for s in rel_stages
        )
        rel_nv = sum(
            float(n_map[s].get("vcpu_seconds") or (float(n_map[s]["wall_clock_sec"]) * TOTAL_WORKER_VCPUS))
            for s in rel_stages
        )
        rel_row: dict[str, Any] = {
            "stage_id": "TOTAL_RELATIONAL_1_2A_3",
            "base_wall_sec": round(rel_bw, 2),
            "nqe_wall_sec": round(rel_nw, 2),
            "nqe_vs_base_speedup_x": round(rel_bw / max(rel_nw, 1e-6), 2),
            "nqe_time_reduction_pct": round(((rel_bw - rel_nw) / max(rel_bw, 1e-6)) * 100.0, 1),
            "base_vcpu_sec": round(rel_bv, 2),
            "nqe_vcpu_sec": round(rel_nv, 2),
        }
        if all(s in std_by_stage for s in rel_stages):
            rel_sw = sum(float(std_by_stage[s]["wall_clock_sec"]) for s in rel_stages)
            rel_row["std_wall_sec"] = round(rel_sw, 2)
            rel_row["base_vs_std_speedup_x"] = round(rel_sw / max(rel_bw, 1e-6), 2)
            rel_row["nqe_vs_std_speedup_x"] = round(rel_sw / max(rel_nw, 1e-6), 2)
        rows.append(rel_row)
    return pd.DataFrame(rows)


def compute_breakeven_table(
    measured_nqe_vs_base_speedup_pct: float,
    measured_nqe_vs_std_speedup_pct: float | None = None,
) -> pd.DataFrame:
    '''Evaluate +$0.0025/vCPU-hr Lightning surcharge break-even across pricing profiles.'''
    profiles = [
        ("100% On-Demand (n2d + SSD)", GCE_N2D_VCPU_HR_ONDEMAND + DATAPROC_FEE_VCPU_HR + LOCAL_SSD_VCPU_HR),
        ("1-Yr CUD (n2d + SSD)", GCE_N2D_VCPU_HR_1YR_CUD + DATAPROC_FEE_VCPU_HR + LOCAL_SSD_VCPU_HR),
        ("3-Yr CUD (n2d + SSD)", GCE_N2D_VCPU_HR_3YR_CUD + DATAPROC_FEE_VCPU_HR + LOCAL_SSD_VCPU_HR),
        (
            "15% 1-Yr CUD Primary + 85% Spot EFM",
            (0.15 * GCE_N2D_VCPU_HR_1YR_CUD) + (0.85 * GCE_N2D_VCPU_HR_SPOT) + DATAPROC_FEE_VCPU_HR + LOCAL_SSD_VCPU_HR,
        ),
        (
            "15% 1-Yr CUD + 85% Spot (Compute-Only Ref)",
            (0.15 * GCE_N2D_VCPU_HR_1YR_CUD) + (0.85 * GCE_N2D_VCPU_HR_SPOT),
        ),
    ]
    rows: list[dict[str, Any]] = []
    measured = (
        measured_nqe_vs_std_speedup_pct
        if measured_nqe_vs_std_speedup_pct is not None
        else measured_nqe_vs_base_speedup_pct
    )
    for name, std_rate in profiles:
        lgt_rate = std_rate + LIGHTNING_SURCHARGE_VCPU_HR
        req_speedup_pct = (LIGHTNING_SURCHARGE_VCPU_HR / std_rate) * 100.0
        req_time_cut_pct = (LIGHTNING_SURCHARGE_VCPU_HR / lgt_rate) * 100.0
        rows.append(
            {
                "pricing_profile": name,
                "std_rate_per_vcpu_hr": round(std_rate, 5),
                "lightning_rate_per_vcpu_hr": round(lgt_rate, 5),
                "surcharge_premium_pct": round(req_speedup_pct, 1),
                "required_breakeven_speedup_pct": round(req_speedup_pct, 1),
                "required_time_reduction_pct": round(req_time_cut_pct, 1),
                "measured_relational_speedup_pct": round(measured, 1),
                "net_cost_positive": measured >= req_speedup_pct,
            }
        )
    return pd.DataFrame(rows)


penalty_df = compute_stage2b_penalty(base_records + nqe_records)
if penalty_df is not None:
    print("=== Stage 2A Vectorized vs. Stage 2B Scalar @F.udf Penalty ===")
    display(penalty_df)
else:
    print("Stage 2B skipped (RUN_PYTHON_UDF_STAGE=false).")

std_baseline = load_optional_standard_baseline()
speedup_df = build_speedup_table(base_records, nqe_records, std_baseline)
print("=== Stage-by-Stage Speedup Summary ===")
if not std_baseline:
    print("(Note: spark_standard baseline not yet in live_runs/benchmark_runs.jsonl for this SCALE_PRESET; showing Base vs NQE.)")
display(speedup_df)

rel_summary = speedup_df[speedup_df["stage_id"] == "TOTAL_RELATIONAL_1_2A_3"].iloc[0]
nqe_vs_base_pct = (float(rel_summary["nqe_vs_base_speedup_x"]) - 1.0) * 100.0
nqe_vs_std_pct = (
    (float(rel_summary["nqe_vs_std_speedup_x"]) - 1.0) * 100.0
    if "nqe_vs_std_speedup_x" in rel_summary
    else None
)
breakeven_df = compute_breakeven_table(nqe_vs_base_pct, nqe_vs_std_pct)
print("=== Lightning Engine +$0.0025/vCPU-hr Break-Even Analysis ===")
display(breakeven_df)


def plot_lightning_diagnostics(spd_df: pd.DataFrame, brk_df: pd.DataFrame) -> None:
    '''Render stage latency bar chart and break-even speedup threshold comparison.'''
    stage_rows = spd_df[spd_df["stage_id"] != "TOTAL_RELATIONAL_1_2A_3"].copy()
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    x = range(len(stage_rows))
    width = 0.28 if "std_wall_sec" in stage_rows.columns else 0.36
    if "std_wall_sec" in stage_rows.columns:
        axes[0].bar([i - width for i in x], stage_rows["std_wall_sec"], width=width, label="spark_standard")
        axes[0].bar(x, stage_rows["base_wall_sec"], width=width, label="spark_lightning_base")
        axes[0].bar([i + width for i in x], stage_rows["nqe_wall_sec"], width=width, label="spark_lightning_nqe")
    else:
        axes[0].bar([i - width / 2 for i in x], stage_rows["base_wall_sec"], width=width, label="spark_lightning_base")
        axes[0].bar([i + width / 2 for i in x], stage_rows["nqe_wall_sec"], width=width, label="spark_lightning_nqe")

    axes[0].set_xticks(list(x))
    axes[0].set_xticklabels(stage_rows["stage_id"], rotation=20, ha="right")
    axes[0].set_ylabel("Wall-Clock Seconds")
    axes[0].set_title(f"Stage Latency: Base vs. NQE ({SCALE_PRESET})")
    axes[0].legend()
    axes[0].grid(axis="y", alpha=0.3)

    y_pos = range(len(brk_df))
    axes[1].barh(y_pos, brk_df["required_breakeven_speedup_pct"], label="Required Break-Even Speedup %")
    measured_val = float(brk_df["measured_relational_speedup_pct"].iloc[0])
    axes[1].axvline(
        measured_val,
        color="crimson",
        linestyle="--",
        linewidth=2,
        label=f"Measured Speedup ({measured_val:.1f}%)",
    )
    axes[1].set_yticks(list(y_pos))
    axes[1].set_yticklabels(brk_df["pricing_profile"])
    axes[1].set_xlabel("Speedup % Required to Offset +$0.0025/vCPU-hr")
    axes[1].set_title("Lightning Engine Surcharge Break-Even Thresholds")
    axes[1].legend()
    axes[1].grid(axis="x", alpha=0.3)

    plt.tight_layout()
    plt.show()


plot_lightning_diagnostics(speedup_df, breakeven_df)